## Datos

#### Bibliotecas

In [ ]:
import os
import math
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle, Patch
from matplotlib.ticker import FuncFormatter
from matplotlib.colors import LogNorm
from IPython.display import display
from typing import Dict, List, Tuple, Optional

#### Cargar Fichero

In [ ]:
# Análisis y gráficas de consumo de tiempo y RAM
# - Carga (si no existe la variable `df` ya cargada)
# - Detección automática de columnas candidatas para tiempo y RAM
# - Definición de claves de agrupamiento en `group_keys` (edítalas según necesites)
# - Agrupación con funciones en un diccionario y creación de un DataFrame agrupado
# - Dos funciones de trazado: plot_time() y plot_ram()

sns.set(style="whitegrid")

# Archivo por defecto (editar si lo cambias de sitio)
file = "OpenFHE_results.csv"

# Si la celda anterior ya creó 'df', la reutilizamos; si no, la cargamos
df = pd.read_csv(file, sep=',', low_memory=False)

print("Columnas detectadas:")
print(list(df.columns))

#### Asignar tipos a las columnas

In [ ]:
# Asignar tipos correctos si es necesario (editar según el CSV)
col_types = {
    'algParallelism': str,
    'm': int,
    'k': int,
    'n': int,
    'algorithm': str,
    'reIteraciones': int,
    'repeticion': int,
    'ringDim': int,
    'batchSize': int,
    'multDepth': int,
    'dcrtBits': int,
    'firstMod': int,
    'securityLevel': str,
    'scalingTechnique': str,
    'encryptionTechnique': str,
    'keySwitchTechnique': str,
    'multiplicationTechnique': str,
    'proxyReEncryptionMode': str,
    'multipartyMode': str,
    'executionMode': str,
    'decryptionNoiseMode': str,
    'modulus': str,
    'ctxA_size': int,
    'ctxB_size': int,
    'ctxC_size': int,
    'ctx_noiseScaleDeg': int,
    'ctx_scalingFactor': str,
    'ctx_hopLevel': int,
    'ctx_level_min': float,
    'ctx_level_max': float,
    'ctx_level_mean': float,
    'ctx_level_median': float,
    'ctx_level_p10': float,
    'ctx_level_p25': float,
    'ctx_level_p75': float,
    'ctx_level_p90': float,
    'ctx_level_std': float,
    'ptx_length': float,
    'ptx_logError_min': float,
    'ptx_logError_max': float,
    'ptx_logError_mean': float,
    'ptx_logError_median': float,
    'ptx_logError_p10': float,
    'ptx_logError_p25': float,
    'ptx_logError_p75': float,
    'ptx_logError_p90': float,
    'ptx_logError_std': float,
    'ptx_logPrecision_min': float,
    'ptx_logPrecision_max': float,
    'ptx_logPrecision_mean': float,
    'ptx_logPrecision_median': float,
    'ptx_logPrecision_p10': float,
    'ptx_logPrecision_p25': float,
    'ptx_logPrecision_p75': float,
    'ptx_logPrecision_p90': float,
    'ptx_logPrecision_std': float,
    'errorAbs_min': float,
    'errorAbs_max': float,
    'errorAbs_mean': float,
    'errorAbs_median': float,
    'errorAbs_p10': float,
    'errorAbs_p25': float,
    'errorAbs_p75': float,
    'errorAbs_p90': float,
    'errorAbs_std': float,
    'errorRel_min': float,
    'errorRel_max': float,
    'errorRel_mean': float,
    'errorRel_median': float,
    'errorRel_p10': float,
    'errorRel_p25': float,
    'errorRel_p75': float,
    'errorRel_p90': float,
    'errorRel_std': float,
    'errorSmape_min': float,
    'errorSmape_max': float,
    'errorSmape_mean': float,
    'errorSmape_median': float,
    'errorSmape_p10': float,
    'errorSmape_p25': float,
    'errorSmape_p75': float,
    'errorSmape_p90': float,
    'errorSmape_std': float,
    'dur_Test': float,
    'dur_Params': float,
    'dur_CryptoContext': float,
    'dur_KeyGeneration': float,
    'dur_Preprocess': float,
    'dur_Encode': float,
    'dur_Encrypt': float,
    'dur_MatrixMultiplication': float,
    'dur_Decrypt': float,
    'dur_Decode': float,
    'dur_Postprocess': float,
    'dur_All': float,
    'ram_Test': float,
    'ram_Params': float,
    'ram_CryptoContext': float,
    'ram_KeyGeneration': float,
    'ram_Preprocess': float,
    'ram_Encode': float,
    'ram_Encrypt': float,
    'ram_MatrixMultiplication': float,
    'ram_Decrypt': float,
    'ram_Decode': float,
    'ram_Postprocess': float,
    'ram_All': float,
    'exceptions': list
}
for col, dtype in col_types.items():
    if col in df.columns:
        if dtype == list:
            # Suponiendo que las listas están almacenadas como strings en el CSV
            df[col] = df[col].apply(lambda x: x.split(';') if pd.notna(x) else [])
        elif dtype in [int, float]:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(dtype)
        elif dtype == str:
            df[col] = df[col].apply(lambda x: str(x) if pd.notna(x) else '').astype(dtype)
        else:
          df[col] = df[col].astype(dtype)

#### Transformar Valores

In [ ]:
col_transform = {
    'algorithm': {
        'Traditional': 'Naive',
        'HEMatMult': 'JiangKimLauterSong'
    }
}
df.replace(col_transform, inplace=True)

#### Agrupar para Tiempo y Memoria

In [ ]:
# Pon aquí las columnas del CSV que quieres usar como agrupación (ordenadas para el eje x y hue)
group_keys = ['algParallelism', 'm', 'k', 'n', 'algorithm', 'reIteraciones', 'ringDim', 'batchSize', 'multDepth', 'dcrtBits', 'firstMod', 'securityLevel', 'scalingTechnique', 'encryptionTechnique', 'keySwitchTechnique', 'multiplicationTechnique', 'proxyReEncryptionMode', 'multipartyMode', 'executionMode', 'decryptionNoiseMode', 'modulus', 'ctxA_size', 'ctxB_size', 'ctxC_size']
# --------------------------------------------------------
print('\nColumnas de agrupación:', group_keys)

# Heurística para detectar columnas de tiempo y RAM (editar si es necesario)
time_candidates = [c for c in df.columns if 'dur_' in c.lower()]
ram_candidates = [c for c in df.columns if 'ram_' in c.lower()]

print('Columnas de tiempo:', time_candidates)
print('Columnas de RAM:   ', ram_candidates)

# Validaciones básicas: filtrar group_keys por las columnas que realmente existen y avisar
valid_group_keys = [k for k in group_keys if k in df.columns]
missing_keys = [k for k in group_keys if k not in df.columns]
if missing_keys:
    print(f"Advertencia: las siguientes columnas de `group_keys` no existen y serán ignoradas: {missing_keys}")
if len(valid_group_keys) == 0:
    print("Advertencia: no quedan columnas de agrupación válidas. La agregación se hará sobre todo el dataset.")

# --- Definir funciones de agregación ---
def p10(g):
    return np.percentile(g, 10)
def p25(g):
    return np.percentile(g, 25)
def p75(g):
    return np.percentile(g, 75)
def p90(g):
    return np.percentile(g, 90)
agg_spec = {}
for col in time_candidates:
    agg_spec[col] = ['mean', 'median', 'min', 'max', p10, p25, p75, p90, 'std']
for col in ram_candidates:
    agg_spec[col] = ['mean', 'median', 'min', 'max', p10, p25, p75, p90, 'std']

if not agg_spec:
    raise RuntimeError('No se han detectado columnas de tiempo ni RAM. Ajusta las heurísticas o renombra columnas en el CSV.')

# Eliminar filas con excepciones (si existe la columna), si no existe, usar el dataframe completo
initial_rows = df.shape[0]
if 'exceptions' in df.columns:
    df_filtered = df[df['exceptions'].isna() | df['exceptions'].str.len() == 0].copy()
    filtered_rows = df_filtered.shape[0]
    print(f"Filas con excepciones eliminadas: {initial_rows - filtered_rows} (de {initial_rows} a {filtered_rows})")
else:
    df_filtered = df.copy()
    print("Columna 'exceptions' no encontrada: no se han filtrado filas por excepciones.")

# Agrupar y agregar (si hay claves válidas) o agregar sobre todo el DataFrame si no las hay
if len(valid_group_keys) > 0:
    grouped = df_filtered.groupby(valid_group_keys).agg(agg_spec)
    # Aplanar columnas MultiIndex resultante
    grouped.columns = ['.'.join(col).strip() for col in grouped.columns.values]
    grouped = grouped.reset_index()
else:
    # agregación global -> resultado de una fila
    grouped_temp = df_filtered.agg(agg_spec)
    # grouped_temp será una Serie con MultiIndex en el índice; convertir a DataFrame de una fila
    grouped = pd.DataFrame([grouped_temp])
    # aplanar nombres de columna
    grouped.columns = ['.'.join(col).strip() if isinstance(col, tuple) else str(col) for col in grouped.columns.values]

# crear columna combinada "m x k x n" como string y colocarla al inicio
if all(c in grouped.columns for c in ['m', 'k', 'n']):
  grouped.insert(0, 'tamaño', grouped['m'].astype(str) + 'x' + grouped['k'].astype(str) + 'x' + grouped['n'].astype(str))
else:
  grouped.insert(0, 'tamaño', '')

ramThreshold = 2**42  # 4 TB en bytes
for col in grouped.columns:
    #transformar las duraciones de ms a s
    if col.startswith('dur_'):
        grouped[col] = grouped[col] / 1000.0
    if col.startswith('ram_'):
        # nulificar valores por encima de 4 TB
        grouped[col] = pd.to_numeric(grouped[col], errors='coerce')
        grouped.loc[grouped[col] > ramThreshold, col] = np.nan

# Filtrar solo matrices cuadradas (m == k == n)
if all(c in grouped.columns for c in ['m', 'k', 'n']):
  before_count = len(grouped)
  grouped_cuad = grouped[(grouped['m'] == grouped['k']) & (grouped['k'] == grouped['n'])].reset_index(drop=True)
  after_count = len(grouped_cuad)
  print(f"Filas tras filtrar cuadradas (m==k==n): {after_count} (de {before_count})")
else:
  print("No se puede filtrar por cuadradas: faltan las columnas 'm', 'k' o 'n' en `grouped`.")

print('\nEjemplo de filas agrupadas:')
from IPython.display import display
display(grouped.head())

#### Agrupar para error

In [ ]:
# Pon aquí las columnas del CSV que quieres usar como agrupación (ordenadas para el eje x y hue)
group_keys = ['algParallelism', 'm', 'k', 'n', 'algorithm', 'reIteraciones', 'ringDim', 'batchSize', 'multDepth', 'dcrtBits', 'firstMod', 'securityLevel', 'scalingTechnique', 'encryptionTechnique', 'keySwitchTechnique', 'multiplicationTechnique', 'proxyReEncryptionMode', 'multipartyMode', 'executionMode', 'decryptionNoiseMode', 'modulus', 'ctxA_size', 'ctxB_size', 'ctxC_size']
# --------------------------------------------------------
print('\nColumnas de agrupación:', group_keys)

valid_group_keys = [k for k in group_keys if k in df.columns]
missing_keys = [k for k in group_keys if k not in df.columns]
if missing_keys:
    print(f"Advertencia: las siguientes columnas de `group_keys` no existen y serán ignoradas: {missing_keys}")
if len(valid_group_keys) == 0:
    print("Advertencia: no quedan columnas de agrupación válidas. La agregación se hará sobre todo el dataset.")

# Heurística para detectar columnas de tiempo y RAM (editar si es necesario)
err_candidates = ['errorAbs', 'errorRel', 'errorSmape']

print('Columnas de error:', err_candidates)


# --- Definir funciones de agregación ---
def std(err_can, rows):
    #display(rows.head())
    if 'count' in rows.columns:
        ns = rows['count'].astype(float)
    elif all(c in rows.columns for c in ['m','n']):
        ns = (rows['m'].astype(float) * rows['n'].astype(float))
    else:
        # si no hay información de tamaño, asumimos que cada fila representa el mismo número de muestras
        # y pedimos al usuario que ajuste la variable 'assumed_n' abajo si procede.
        assumed_n = None
        if assumed_n is None:
            raise RuntimeError("No se ha detectado 'count' ni 'm'/'n' en df; define 'assumed_n' con el número de muestras por subgrupo.")
        ns = np.full(len(rows), assumed_n, dtype=float)
    
    N = ns.sum()
    #print(f'Número total de observaciones N = {int(N)} (G={len(rows)} subgrupos)')

    if err_can + "_mean" not in rows.columns or err_can + "_std" not in rows.columns:
        print(f'Advertencia: no se encontraron columnas mean/std para {err_can} (buscando {err_can + "_mean"}/{err_can + "_std"}). Se omite.')
        return pd.Series([np.nan], index=[err_can + '.std'])

    means = rows[err_can + '_mean'].astype(float).fillna(0.0)
    stds = rows[err_can + '_std'].astype(float).fillna(0.0)
    vars_ = stds ** 2
    
    # media global (ponderada por tamaños). Si todos ns iguales, equivale a mean of means
    global_mean = (ns * means).sum() / N
    # S_within: suma de sumas de cuadrados dentro de grupos (muestral)
    # asumimos que 'stds' es std muestral (ddof=1) tal y como produce pandas .std() por defecto
    S_within = ((np.asarray(ns) - 1.0) * vars_).sum()
    # S_between: variación debida a diferencias entre medias de subgrupo y media global
    S_between = (ns * (means - global_mean) ** 2).sum()
    SS_total = S_within + S_between
    if N > 1:
        var_global = SS_total / (N - 1.0)
    else:
        var_global = np.nan
    std_global = np.sqrt(var_global)
    #print(f'Global std de {err_can}: {std_global} (mean={global_mean}, S_within={S_within}, S_between={S_between}, SS_total={SS_total})')
    return pd.Series([std_global], index=[f"{err_can}.std"])
def percentiles(err_can, rows):
    agrupado = []
    for ele in ['min', 'p10', 'p25', 'median', 'p75', 'p90', 'max']:
        col_name = f"{err_can}_{ele}"
        if col_name in rows.columns:
            agrupado += rows[col_name].astype(float).values.tolist()
        else:
            print(f'Advertencia: no se encontró la columna {col_name} para percentiles de {err_can}. Se usará NaN.')
            agrupado += np.full(len(rows), np.nan).tolist()
    return pd.Series([np.percentile(agrupado, 10), np.percentile(agrupado, 25), np.median(agrupado), np.percentile(agrupado, 75), np.percentile(agrupado, 90)], index=[f"{err_can}.p10", f"{err_can}.p25", f"{err_can}.median", f"{err_can}.p75", f"{err_can}.p90"])
# Aproximación de percentiles pooled a partir de summaries por subgrupo
# Método: reconstrucción de CDF por tramos entre quantiles disponibles (min,p10,p25,median,p75,p90,max)
# y muestreo sintético proporcional al tamaño del subgrupo (si está disponible).
# Devuelve percentiles (p10,p25,p50,p75,p90) aproximados para el conjunto global.
def sample_from_quantiles(q_vals: List[float], q_probs: List[float], n_samples: int, rng=None) -> np.ndarray:
    """Genera `n_samples` aproximados a partir de una CDF lineal entre puntos (q_probs, q_vals).
    q_vals: valores de la variable en los quantiles (ej. [min,p10,p25,median,p75,p90,max])
    q_probs: probabilidades asociadas (ej. [0,0.1,0.25,0.5,0.75,0.9,1.0])
    Se interpola la CDF por tramos y se aplica muestreo inverso.
    """
    if rng is None:
        rng = np.random.default_rng()
    q_probs = np.asarray(q_probs, dtype=float)
    q_vals = np.asarray(q_vals, dtype=float)
    # limpiar NaNs: si faltan tramos, rellenar con extremos repetidos
    mask = ~np.isnan(q_vals)
    if mask.sum() < 2:
        return np.full(n_samples, np.nan)
    # lineal interpolation of the inverse CDF via piecewise segments
    # construir arrays de tramos válidos
    probs = q_probs[mask]
    vals = q_vals[mask]
    # asegurar que probs esté estrictamente creciente
    if not np.all(np.diff(probs) > 0):
        # ordenar por probs
        order = np.argsort(probs)
        probs = probs[order]
        vals = vals[order]
    u = rng.random(n_samples)
    samples = np.empty(n_samples, dtype=float)
    # para cada muestra, localizar el tramo y linealmente interpolar
    inds = np.searchsorted(probs, u, side='right')
    for i in range(len(u)):
        idx = inds[i]
        if idx == 0:
            samples[i] = vals[0]
        elif idx >= len(probs):
            samples[i] = vals[-1]
        else:
            # tramo [probs[idx-1], probs[idx]] -> vals[idx-1], vals[idx]
            p0, p1 = probs[idx-1], probs[idx]
            v0, v1 = vals[idx-1], vals[idx]
            if p1 == p0:
                samples[i] = v0
            else:
                t = (u[i] - p0) / (p1 - p0)
                samples[i] = v0 + t * (v1 - v0)
    return samples
def pooled_percentiles_from_grouped_quantiles(err_prefix: str, grouped_df: pd.DataFrame, total_samples: int = 5000, count_col: Optional[str] = None, seed: Optional[int] = None) -> Dict[str, float]:
    """Calcula percentiles pooled (p10,p25,p50,p75,p90) aproximados a partir de un DataFrame `grouped_df`
    que contiene para cada subgrupo las columnas: {err_prefix}_min, {err_prefix}_p10, {err_prefix}_p25, {err_prefix}_median, {err_prefix}_p75, {err_prefix}_p90, {err_prefix}_max.
    Si existe `count_col` (por ejemplo 'count') se usa como pesos; si no, se busca m*n, si tampoco existe, se pondera uniformemente.
    Devuelve un diccionario con las percentiles y con una aproximación naive para comparar.
    """
    rng = np.random.default_rng(seed)
    q_keys = ['min','p10','p25','median','p75','p90','max']
    prob_keys = [0.0, 0.10, 0.25, 0.50, 0.75, 0.90, 1.0]
    # comprobar existencia de columnas y recopilar por fila los quantiles
    rows = []
    for idx, row in grouped_df.iterrows():
        qvals = []
        missing = False
        for k in q_keys:
            c = f"{err_prefix}_{k}"

            if c in grouped_df.columns:
                qvals.append(row[c])
            else:
                qvals.append(np.nan)
                missing = True
        # peso (ns): preferir count_col, si no m*n, si no 1
        if count_col and (count_col in grouped_df.columns):
            try:
                w = float(row[count_col])
            except Exception:
                w = 0.0
        elif 'm' in grouped_df.columns and 'n' in grouped_df.columns and not pd.isna(row.get('m')) and not pd.isna(row.get('n')):
            try:
                w = float(row['m']) * float(row['n'])
            except Exception:
                w = 1.0
        else:
            w = 1.0
        rows.append((qvals, w))
    # eliminar filas sin datos válidos (all NaN) o peso cero
    rows = [(q,w) for (q,w) in rows if (not np.all(np.isnan(q))) and (w is not None) and (w > 0)]
    if len(rows) == 0:
        raise RuntimeError('No hay subgrupos con quantiles válidos para calcular percentiles pooled.')
    # calcular proporción de muestras por grupo
    weights = np.array([w for (_,w) in rows], dtype=float)
    weights = weights / weights.sum()
    # asignar muestras por grupo proporcionalmente (al menos 2 por grupo)
    samples_per_group = np.maximum(2, np.floor(weights * total_samples).astype(int))
    # ajustar para que la suma sea total_samples
    surplus = int(total_samples - samples_per_group.sum())
    if surplus > 0:
        # repartir el surplus según los restos de la multiplicación real
        remainders = (weights * total_samples) - np.floor(weights * total_samples)
        order = np.argsort(remainders)[::-1]
        i = 0
        while surplus > 0:
            samples_per_group[order[i % len(order)]] += 1
            surplus -= 1
            i += 1
    elif surplus < 0:
        # quitar algunos si hemos excedido (caso raro por floor+min)
        deficit = -surplus
        order = np.argsort(samples_per_group)
        i = 0
        while deficit > 0 and i < len(order):
            take = min(deficit, samples_per_group[order[i]] - 1)
            samples_per_group[order[i]] -= take
            deficit -= take
            i += 1
    # muestrear por cada grupo usando la reconstrucción CDF por tramos
    pooled_samples = []
    for (qvals, w), nsamp in zip(rows, samples_per_group):
        if nsamp <= 0:
            continue
        s = sample_from_quantiles(qvals, prob_keys, nsamp, rng=rng)
        pooled_samples.append(s)
    if len(pooled_samples) == 0:
        raise RuntimeError('No se pudieron generar muestras para percentiles pooled (todos NaN?).')
    pooled = np.concatenate(pooled_samples)
    # eliminar NaNs resultantes
    pooled = pooled[~np.isnan(pooled)]
    if pooled.size == 0:
        raise RuntimeError('Tras eliminar NaNs no quedan muestras para calcular percentiles.')
    return pd.Series([np.percentile(pooled, 10), np.percentile(pooled, 25), np.median(pooled), np.percentile(pooled, 75), np.percentile(pooled, 90)], index=[f"{err_prefix}.p10", f"{err_prefix}.p25", f"{err_prefix}.median", f"{err_prefix}.p75", f"{err_prefix}.p90"])

agg_spec = {}
for col in err_candidates:
    agg_spec[col + "_min"] = ['min']
    agg_spec[col + "_max"] = ['max']
    agg_spec[col + "_mean"] = ['mean']

if not agg_spec:
    raise RuntimeError('No se han detectado columnas de tiempo ni RAM. Ajusta las heurísticas o renombra columnas en el CSV.')

# Eliminar filas con excepciones (si existe la columna), si no existe, usar el dataframe completo
initial_rows = df.shape[0]
if 'exceptions' in df.columns:
    df_filtered = df[df['exceptions'].isna() | df['exceptions'].str.len() == 0].copy()
    filtered_rows = df_filtered.shape[0]
    print(f"Filas con excepciones eliminadas: {initial_rows - filtered_rows} (de {initial_rows} a {filtered_rows})")
else:
    df_filtered = df.copy()
    print("Columna 'exceptions' no encontrada: no se han filtrado filas por excepciones.")

# Agrupar y agregar (si hay claves válidas) o agregar sobre todo el DataFrame si no las hay
if len(valid_group_keys) > 0:
    grouped_err = df_filtered.groupby(valid_group_keys).agg(agg_spec)
    # Aplanar columnas MultiIndex resultante
    grouped_err.columns = ['.'.join(col).strip() for col in grouped_err.columns.values]
    for err_can in err_candidates:
        grouped_err = grouped_err.join(df_filtered.groupby(valid_group_keys).apply(lambda rows: std(err_can, rows), include_groups=True))
        #grouped_err = grouped_err.join(df_filtered.groupby(valid_group_keys).apply(lambda rows: percentiles(err_can, rows), include_groups=True))
        grouped_err = grouped_err.join(df_filtered.groupby(valid_group_keys).apply(lambda rows: pooled_percentiles_from_grouped_quantiles(err_can, rows), include_groups=True))
    grouped_err = grouped_err.reset_index()
else:
    # agregación global -> resultado de una fila
    grouped_temp = df_filtered.agg(agg_spec)
    # grouped_temp será una Serie con MultiIndex en el índice; convertir a DataFrame de una fila
    grouped_err = pd.DataFrame([grouped_temp])
    # aplanar nombres de columna
    grouped_err.columns = ['.'.join(col).strip() if isinstance(col, tuple) else str(col) for col in grouped_err.columns.values]

for col in grouped_err.columns:
    if col.startswith('error') and '_' in col:
      grouped_err.rename(columns={col: col.split('_')[0] + '.' + col.split('.')[1]}, inplace=True)

# crear columna combinada "m x k x n" como string y colocarla al inicio
if all(c in grouped_err.columns for c in ['m', 'k', 'n']):
  grouped_err.insert(0, 'tamaño', grouped_err['m'].astype(str) + 'x' + grouped_err['k'].astype(str) + 'x' + grouped_err['n'].astype(str))
else:
  grouped_err.insert(0, 'tamaño', '')

# Filtrar solo matrices cuadradas (m == k == n)
if all(c in grouped_err.columns for c in ['m', 'k', 'n']):
  before_count = len(grouped_err)
  grouped_err_cuad = grouped_err[(grouped_err['m'] == grouped_err['k']) & (grouped_err['k'] == grouped_err['n'])].reset_index(drop=True)
  after_count = len(grouped_err_cuad)
  print(f"Filas tras filtrar cuadradas (m==k==n): {after_count} (de {before_count})")
else:
  print("No se puede filtrar por cuadradas: faltan las columnas 'm', 'k' o 'n' en `grouped`.")

print('\nEjemplo de filas agrupadas:')
display(grouped_err.head())

## Gráficas

### Config

In [ ]:
fig_dir = "./fig"
os.makedirs(fig_dir, exist_ok=True)

sup_map = {'0':'⁰','1':'¹','2':'²','3':'³','4':'⁴','5':'⁵','6':'⁶','7':'⁷','8':'⁸','9':'⁹','-':'⁻'}
def format_exponent(exp: int) -> str:
  return ''.join(sup_map[d] for d in str(exp))

units_mem = {'B': 2**0, 'KB': 2**10, 'MB': 2**20, 'GB': 2**30, 'TB': 2**40, 'PB': 2**50, 'EB': 2**60, 'ZB': 2**70, 'YB': 2**80}
mem_unit = 'GB'  # unidad para mostrar memoria

acronym_alg = {
  "Naive": "Naive",
  "RowsXCols": "RxC",
  "HaleviShoup": "HaSh",
  "JiangKimLauterSong": "JKLS",
  "RizomiliotisTriakosia": "RiTr",
  "Strassen1x1x1": "Str1",
  "StrassenRizomiliotisTriakosia": "StrRiTr"
}
alg_order = ["Naive", "RowsXCols", "HaleviShoup", "JiangKimLauterSong", "RizomiliotisTriakosia", "Strassen1x1x1", "StrassenRizomiliotisTriakosia"]
algParallelisms=["MultiThreadFor(96)", "SingleThread"]

markers = ['.', 's', 'd', 'P', '*', 'X', 'h', '+', 'x', 'o']

colTitleLabels = {
  'dur': ['Duration', 'Duration (s)', 'duration', 90],
  'ram': ['Memory', 'Memory usage', 'memory', 0],
  'ramdur': ['Memory*Duration', 'Memory*Duration (GB*min)', 'memoryDuration', 0],
  'errorAbs': ['Absolute Error', 'Absolute Error', 'errorAbs', 0],
  'errorRel': ['Relative Error', 'Relative Error', 'errorRel', 0],
  'errorSmape': ['SMAPE', 'SMAPE', 'SMAPE', 0]
}

grafica = '.mean'  # '.mean' o '.median'

### Tipos

In [ ]:
def pivot_for(colname, grouped, sizes, algorithms, index_name='tamaño'):
  try:
    if index_name not in grouped.columns:
      raise RuntimeError(f'La columna {index_name} no existe en el DataFrame agrupado.')
    if type(grouped[index_name].iloc[0]) != type(sizes[0]):
      gp = grouped.copy()
      gp[index_name] = gp[index_name].astype(type(sizes[0]))
    else:
      gp = grouped
    pv = gp.pivot_table(index=index_name, columns='algorithm', values=colname, aggfunc='mean')
    pv = pv.reindex(sizes)
    return pv
  except Exception:
    return pd.DataFrame(index=sizes, columns=algorithms)

def fmt_time_seconds(x, pos):
  # x viene en segundos (float). Devuelve etiquetas legibles
  x = float(x)
  if x >= 3600:
    h = x // 3600
    m = (x % 3600) // 60
    return f"10{format_exponent(int(math.log10(x)))}\n({int(h)}:{int(m):02d}:{int(x)%60:02d})"
  if x >= 60:
    m = x // 60
    return f"10{format_exponent(int(math.log10(x)))}\n(0:{int(m):02d}:{int(x)%60:02d})"
  if x >= 1:
    return f"10{format_exponent(int(math.log10(x)))}\n(0:00:{int(x):02d})"
  # para fracciones de segundo
  return f"10{format_exponent(int(math.log10(x)))}\n({int(x*1000)}ms)"

def fmt_memory_bytes(x, pos):
  # x viene en bytes. Devuelve etiquetas legibles
  for label, scale in reversed(units_mem.items()):
    if x >= scale:
      return f"{x/scale:g}{label}"
  return f"{x:g}B"

#### G1: Grafica de lineas con barras(ringDim) y marks(multDeph) y desplazada

In [ ]:
def plot_g1_legend(grouped,
                   alg_order=alg_order,
                   markers=markers,
                   colsAlpha=0.25,
                   figsize=(2,2),
                   fun_alg_palette=lambda alg: sns.color_palette('tab10', n_colors=max(3, len(alg))),
                   axO=None,
                   formatSave=None):
  # si no se pasa un eje, crear una figura dedicada solo para las leyendas
  if axO is None:
    fig, ax = plt.subplots(figsize=figsize)
    # ocultar todo el eje para que la figura contenga únicamente las leyendas
    ax.axis('off')
  else:
    ax = axO
    fig = ax.figure

  # Construir handles para leyenda de algoritmos (líneas) + patch para barras
  algorithms = sorted(grouped['algorithm'].dropna().unique(), key=lambda x: alg_order.index(x)) if 'algorithm' in grouped.columns else []
  if 'multDepth' in grouped.columns:
    mult_vals = sorted(grouped['multDepth'].dropna().unique())
  else:
    mult_vals = []
  # marcadores ordenados de más común a menos y lo más distintos posible
  marker_map = {v: markers[i % len(markers)] for i, v in enumerate(mult_vals)}
  alg_palette = fun_alg_palette(algorithms)
  alg_colors = {alg: alg_palette[i % len(alg_palette)] for i, alg in enumerate(algorithms)}
  line_handles = []
  for alg in algorithms:
    # cuadrado semi-transparente (sin borde) y línea superpuesta
    line_handles.append(
      Line2D(
        [0], [0],
        color=tuple(alg_colors[alg]) + (1.0,),  # línea opaca
        lw=2,
        marker='s',
        markerfacecolor=tuple(alg_colors[alg]) + (colsAlpha,),  # cuadrado con alpha
        markeredgecolor='none',
        markersize=15,
      )
    )
  alg_labels = [str(alg) for alg in algorithms]

  # Crear y colocar la(s) leyenda(s).
  # Si trabajamos con un eje provisto, las leyendas se adjuntan al eje (y se añade la primera como artista
  # para poder tener una segunda leyenda separada). Si no hay eje (figura solo de leyendas), usamos
  # fig.legend para situarlas en la figura.
  if axO is not None:
    legend1 = ax.legend(
      line_handles,
      alg_labels,
      loc='upper left',
      bbox_to_anchor=(1.08, 1.0),
      title='Algorithms',
      frameon=True,
      borderaxespad=0.0
    )
    ax.add_artist(legend1)
  else:
    # colocar la leyenda de algoritmos en la mitad izquierda de la figura
    legend1 = fig.legend(
      line_handles,
      alg_labels,
      loc='lower right',
      bbox_to_anchor=(0.0, 0.0),
      title='Algorithms',
      frameon=True
    )

  # Segunda leyenda: markers por profundidad de multiplicación (multDepth)
  legend2 = None
  if mult_vals:
    marker_handles = []
    marker_labels = []
    for md in mult_vals:
      mkr = marker_map.get(md, 'o')
      marker_handles.append(
        Line2D([0], [0],
               color='gray',
               marker=mkr,
               linestyle='None',
               markerfacecolor='gray',
               markeredgecolor='k',
               markersize=8)
      )
      marker_labels.append(str(md))

    if axO is not None:
      # colocar la segunda leyenda a la derecha de la primera (en coordenadas del eje)
      legend2 = ax.legend(
        marker_handles,
        marker_labels,
        loc='upper left',
        bbox_to_anchor=(1.08, 0.0),
        title='Multiplication depth',
        frameon=True
      )
    else:
      # colocar la segunda leyenda en la mitad derecha de la figura
      legend2 = fig.legend(
        marker_handles,
        marker_labels,
        loc='lower left',
        bbox_to_anchor=(0.0, 0.0),
        title='Multiplication depth',
        frameon=True
      )

  # Si se creó una figura dedicada a las leyendas, mostrar/guardar y cerrar.
  if axO is None:
    if formatSave:
      fname = f"./fig/G1_legend{formatSave}"
      fig.savefig(fname, bbox_inches='tight')
    plt.show()
    plt.close(fig)
  else:
    # devolver las leyendas creadas para uso externo
    return [legend1, legend2] if legend2 is not None else [legend1]

def plot_g1(grouped_ini,
            cols,
            algParallelism=None,
            rectangularMatSize=False,
            yscale=['log'],
            yaxis_major_formatter=None,
            alg_order=alg_order,
            markers=markers,
            colTitleLabels=colTitleLabels,
            formatSave=".png",
            linewidth=2,
            colsAlpha=0.75,
            lineOffset=False,
            figsize=(12, 6),
            fun_alg_palette=lambda alg: sns.color_palette('tab10', n_colors=max(3, len(alg))),
            legend='OUT'):
  if not cols:
    print(f"No se detectaron columnas ({cols}). Revisa la agregación.")
  else:
    grouped = grouped_ini[grouped_ini["algorithm"].isin(alg_order)]
    if algParallelism is not None and 'algParallelism' in grouped.columns:
      if type(algParallelism) == list:
        grouped = grouped[grouped['algParallelism'].isin(algParallelism)]
      else:
        grouped = grouped[grouped['algParallelism'] == algParallelism]
    # construir lista de tamaños ordenada (por m si existe)
    if all(c in grouped.columns for c in ['m','k','n']):
      if rectangularMatSize:
        sizes_df = grouped[['m','k','n']].drop_duplicates().sort_values(['m', 'k', 'n'])
        sizes = (sizes_df['m'].astype(str) + 'x' + sizes_df['k'].astype(str) + 'x' + sizes_df['n'].astype(str)).tolist()
      else:
        #grouped = grouped[grouped['m'] == grouped['k']]
        #grouped = grouped[grouped['k'] == grouped['n']]
        sizes_df = grouped[['m']].drop_duplicates().sort_values(['m'])
        sizes = (sizes_df['m'].astype(str)).tolist()
    else:
      sizes = sorted(grouped['tamaño'].dropna().unique())
    x = np.arange(len(sizes))
    algorithms = sorted(grouped['algorithm'].dropna().unique(), key=lambda x: alg_order.index(x)) if 'algorithm' in grouped.columns else []
    n_alg = max(1, len(algorithms))
    # marker map for multDepth (renombrado en la leyenda a 'Profundidad de multiplicación')
    if 'multDepth' in grouped.columns:
      mult_vals = sorted(grouped['multDepth'].dropna().unique())
    else:
      mult_vals = []
    # marcadores ordenados de más común a menos y lo más distintos posible
    marker_map = {v: markers[i % len(markers)] for i, v in enumerate(mult_vals)}
    # colores y marcadores por algoritmo
    alg_palette = fun_alg_palette(algorithms)
    alg_colors = {alg: alg_palette[i % len(alg_palette)] for i, alg in enumerate(algorithms)}
    base_slot = 0.9 / max(1, n_alg)  # espacio por algoritmo en cada grupo de tamaño

    for col in cols:
      if grouped[col].isna().all() or (grouped[col] == 0).all():
        print(f"Advertencia: la columna {col} contiene solo NaNs y será ignorada.")
        continue
      pivot_mat = pivot_for([col, 'ringDim', 'multDepth'], grouped, sizes, algorithms, index_name=('tamaño' if rectangularMatSize else 'm'))
      fig, ax1 = plt.subplots(figsize=figsize)
      ax2 = ax1.twinx()
      # Asegurar que las líneas (ax1) se dibujen por encima de las barras (ax2)
      ax1.set_zorder(2)
      ax2.set_zorder(1)
      ax1.patch.set_visible(False)
      ax1.xaxis.grid(True, which='major', linestyle='--', alpha=0.6)
      ax1.xaxis.grid(True, which='minor', linestyle='--', alpha=0.2)

      # líneas por algoritmo (durations)
      for i_alg, alg in enumerate(algorithms):
        offsets = (i_alg - (n_alg - 1) / 2.0) * base_slot
        mask = grouped['algorithm'] == alg
        sub_alg = grouped[mask].copy()
        if sub_alg.empty:
          continue
        y = pivot_mat[col][alg].values if alg in pivot_mat[col].columns else np.zeros(len(sizes))
        # build per-point marker list (may contain None)
        markers_list = []
        if 'multDepth' in pivot_mat and alg in pivot_mat['multDepth'].columns:
          markers_list = [marker_map.get(v, None) for v in pivot_mat['multDepth'][alg].values]
        else:
          markers_list = [None] * len(x)
        # draw the line (no per-point marker)
        ax1.plot(
          (x+offsets) if lineOffset else x, y,
          color=tuple(alg_colors[alg]) + (1/3,),
          label=str(alg),
          linewidth=linewidth,
          zorder=5,
        )
        # draw markers individually so each point can have its own marker style
        for xi, mk, yi in zip(x, markers_list, y):
          # skip missing y values
          if yi is None or (isinstance(yi, float) and np.isnan(yi)):
            continue
          if mk is None:
            # no marker for this point
            continue
          ax1.plot(
            (xi+offsets) if lineOffset else xi, yi,
            marker=mk,
            color=tuple(alg_colors[alg]) + (1.0,),
            markerfacecolor=tuple(alg_colors[alg]) + (1.0,),
            markeredgecolor=tuple(alg_colors[alg]) + (1.0,),
            linestyle='None',
            markersize=6,
            zorder=6,
          )
        ax1.set_ylabel(colTitleLabels.get(col.split('_')[0], colTitleLabels.get(col.split('.')[0]))[1])
        if (yscale):
          ax1.set_yscale(yscale[0], base=yscale[1] if len(yscale) > 1 else 10)
        # color y rotación de las etiquetas del eje Y
        try:
          ax1.tick_params(axis='y', labelcolor='black', labelrotation=colTitleLabels.get(col.split('_')[0], colTitleLabels.get(col.split('.')[0]))[3])
        except TypeError:
          # compatibilidad con versiones antiguas de matplotlib
          ax1.tick_params(axis='y', labelcolor='black')
          for lbl in ax1.get_yticklabels():
            lbl.set_rotation(colTitleLabels.get(col.split('_')[0], colTitleLabels.get(col.split('.')[0]))[3])
        if yaxis_major_formatter:
          ax1.yaxis.set_major_formatter(FuncFormatter(yaxis_major_formatter))
        
        # (Opcional) mantener some minor ticks or grid for readability:
        ax1.yaxis.set_minor_formatter(plt.NullFormatter())
        # ajustar grid (si quieres lineas horizontales de ax1):
        ax1.yaxis.grid(True, which='major', linestyle='--', alpha=0.6)
        ax1.yaxis.grid(True, which='minor', linestyle='--', alpha=0.2)

        # barras por (size, alg) en ax2: altura = ringDim (valor), escala del eje en log base 2
        # Dibujar barras con ancho fijo (sin codificar multDepth en el ancho)
        bar_width_fixed = base_slot * 0.95
        for i_alg, alg in enumerate(algorithms):
          offsets = (i_alg - (n_alg - 1) / 2.0) * base_slot
          for xi, size in enumerate(sizes):
            try:
              rd = pivot_mat['ringDim'].loc[size, alg]
            except Exception:
              rd = np.nan
            if pd.isna(rd):
              continue
            center = xi + offsets
            # asegurar valor positivo y no cero para scale log
            h = float(rd) if rd > 0 else 1.0
            # Garantizar que el color tenga el alpha esperado construyendo un RGBA explícito
            ax2.bar(center, h, width=bar_width_fixed, color=alg_colors[alg], alpha=colsAlpha*0.25, align='center', zorder=2)
        # eje derecho en escala log base 2
        try:
          ax2.set_yscale('log', base=2)
        except TypeError:
          # compatibilidad con versiones antiguas
          ax2.set_yscale('log')
        ax2.set_ylabel('Ring Dimension')
        ax2.tick_params(axis='y', labelcolor='black')
        ax2.yaxis.grid(False)

        # X labels
        ax1.set_xticks(x)
        if rectangularMatSize:
          ax1.set_xticklabels(sizes, rotation=45, ha='right')
          ax1.set_xlabel('Matrix rectangular size (m x k x n)')
        else:
          ax1.set_xticklabels(sizes)
          ax1.set_xlabel('Matrix square size (m x m x m)')
      
        extra_artists = []
        if legend == 'IN':
          extra_artists = plot_g1_legend(grouped,
                                        alg_order=alg_order,
                                        markers=markers,
                                        colsAlpha=colsAlpha,
                                        fun_alg_palette=fun_alg_palette,
                                        axO=ax1)
      try:     
        ax1.set_title(f"{algParallelism} - {colTitleLabels.get(col.split('_')[0], colTitleLabels.get(col.split('.')[0]))[0]} — {col.split('.',1)[0].split('_',1)[1]} ({col.split('.',1)[1].capitalize()})")
      except Exception:
        ax1.set_title(f"{algParallelism} - {'' if col.split('.',1)[0]=='errorSmape' else col.split('.',1)[1].capitalize()+' '}{colTitleLabels.get(col.split('_')[0], colTitleLabels.get(col.split('.')[0]))[0]}")
      # asegurar que las leyendas externas no se corten
      fig.canvas.draw()
      # dejar espacio a la derecha para las leyendas fuera del eje
      fig.subplots_adjust(right=0.75)
      plt.tight_layout()
      # reunir artistas extra (leyendas) para que savefig las incluya
      if formatSave:
        fname = f"{fig_dir}/{algParallelism}_{colTitleLabels.get(col.split('_')[0], colTitleLabels.get(col.split('.')[0]))[2]}_G1{'' if col.split('.')[0]=='errorSmape' else ('_'+((col.split('_')[1].split('.')[0]+'_') if '_' in col else '')+col.split('.')[1])}{formatSave}"
        if extra_artists:
          fig.savefig(fname, bbox_inches='tight', bbox_extra_artists=extra_artists)
        else:
          fig.savefig(fname, bbox_inches='tight')
      plt.show()
      plt.close(fig)
  if legend == 'OUT':
    plot_g1_legend(grouped,
                   alg_order=alg_order,
                   markers=markers,
                   colsAlpha=colsAlpha,
                   fun_alg_palette=fun_alg_palette,
                   axO=None,
                   formatSave=formatSave)

### Tiempo

#### Graficos Tiempo 01

In [ ]:
for algParallelism in algParallelisms:
  plot_g1(grouped_cuad, [col for col in grouped_cuad.columns if col.startswith('dur_') and col.endswith(grafica)], algParallelism=algParallelism, yaxis_major_formatter=fmt_time_seconds)#, lineOffset=True)

### Memoria

#### Graficos Memoria 01

In [ ]:
for algParallelism in algParallelisms:
  plot_g1(grouped_cuad, [col for col in grouped_cuad.columns if col.startswith('ram_') and col.endswith(grafica)], algParallelism=algParallelism, yscale=['log', 2], yaxis_major_formatter=fmt_memory_bytes)

#### Graficos Test Mem*Dur

In [ ]:
grouped_memDur = grouped_cuad.copy()
for col in grouped_cuad.columns:
  if (col.startswith('ram_') and col.endswith(grafica) and ["CryptoContext", "KeyGeneration", "MatrixMultiplication", "All"].__contains__(col.split('_',1)[1].split('.',1)[0])):
    dur_col = f"dur_{col.split('_',1)[1]}"
    if dur_col in grouped_cuad.columns:
      grouped_memDur[f"ramdur_{col.split('_',1)[1]}"] = grouped_cuad[col]/60 * grouped_cuad[dur_col]/2**30  # en GB*min

for algParallelism in algParallelisms:
  plot_g1(grouped_memDur, [col for col in grouped_memDur.columns if col.startswith('ramdur_') and col.endswith(grafica)], algParallelism=algParallelism)

### Error

#### Grafico Error 1

In [ ]:
for algParallelism in algParallelisms:
  plot_g1(grouped_err, [f"errorAbs{grafica}", f"errorRel{grafica}", "errorSmape.mean"], algParallelism=algParallelism)